In [ ]:
!pip install -q -U bitsandbytes peft accelerate datasets
!pip install -q -U bitsandbytes peft accelerate datasets trl

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType


#SETUP & CONFIGURATION


MODEL_ID = "Qwen/Qwen2.5-7B" 
MAX_LEN = 512
BATCH_SIZE = 4       
ACCUM_STEPS = 8      
EPOCHS = 2           
LR = 2e-4            

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    return {"macro_f1": macro_f1}

def main():
    print("Loading Data...")
    train_df = pd.read_csv('../data/train.csv')
    test_df = pd.read_csv('../data/test.csv')

    
    train_split, val_split = train_test_split(
        train_df, test_size=0.1, random_state=42, stratify=train_df["LABEL"]
    )

    
    train_dataset = Dataset.from_pandas(train_split)
    val_dataset = Dataset.from_pandas(val_split)
    test_dataset = Dataset.from_pandas(test_df)

    num_labels = len(train_df["LABEL"].unique())

   
    # TOKENIZATION
    
    print("Loading Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, add_prefix_space=True)

    # 
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    def tokenize_function(examples):
        return tokenizer(
            examples["TEXT"],
            truncation=True,
            max_length=MAX_LEN,
            padding=False 
        )

    print("Tokenizing datasets...")
    train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["TEXT", "__index_level_0__"])
    val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["TEXT", "__index_level_0__"])
    test_tokenized = test_dataset.map(tokenize_function, batched=True, remove_columns=["TEXT"])

    
    train_tokenized = train_tokenized.rename_column("LABEL", "labels")
    val_tokenized = val_tokenized.rename_column("LABEL", "labels")

    
    # 4-BIT QUANTIZATION & MODEL LOADING
    
    print("Loading 7B Model in 4-bit...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map="auto",
    )

    
    model.config.pad_token_id = tokenizer.pad_token_id

    
    # LoRA ADAPTER CONFIGURATION
    
    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    
    # TRAINING VIA HF TRAINER
    
    training_args = TrainingArguments(
        output_dir="./qwen_results",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=ACCUM_STEPS,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="epoch",       
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=False,
        bf16=True,                   
        gradient_checkpointing=True, 
        logging_steps=10,
        report_to="none"
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("Starting Qwen 2.5 Training...")
    trainer.train()

    
    # INFERENCE & SUBMISSION
    
    print("Training complete. Predicting on Test Set...")
    test_predictions = trainer.predict(test_tokenized)

    
    logits = test_predictions.predictions
    final_preds = np.argmax(logits, axis=1)

   
    submission = pd.DataFrame({
        "ID": test_df.index.astype(int) if "ID" not in test_df.columns else test_df["ID"],
        "LABEL": final_preds
    })
    submission.sort_values("ID", inplace=True)
    submission.to_csv("submission_qwen_qlora.csv", index=False)

    
    np.save("qwen_raw_probs.npy", logits)
    print("Saved submission to: submission_qwen_qlora.csv")
    print("Saved probabilities to: qwen_raw_probs.npy")



In [ ]:
if __name__ == "__main__":
    main()

Loading Data...
Loading Tokenizer...
Tokenizing datasets...


Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Loading 7B Model in 4-bit...


...

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


trainable params: 40,391,680 || all params: 7,111,032,320 || trainable%: 0.5680
Starting Qwen 2.5 Training...


Epoch,Training Loss,Validation Loss,Macro F1
1,0.692342,0.151483,0.892706
2,0.386305,0.023251,0.978773


Training complete. Predicting on Test Set...


Saved submission to: submission_qwen_qlora.csv
Saved probabilities to: qwen_raw_probs.npy
